In [1]:
import sys, subprocess, time, warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score, balanced_accuracy_score, recall_score, confusion_matrix, roc_auc_score, roc_curve
from sklearn.linear_model import LogisticRegression
from sklearn.feature_selection import SelectKBest, f_classif
import matplotlib.pyplot as plt

from qiskit.circuit.library import ZZFeatureMap, TwoLocal
from qiskit_aer.primitives import Estimator
from qiskit_machine_learning.neural_networks import EstimatorQNN
from qiskit_machine_learning.algorithms.classifiers import NeuralNetworkClassifier
from qiskit_machine_learning.optimizers import COBYLA

In [2]:
# ===== Parametry do łatwej zmiany =====
data_path   = "countsAll_fixed_07_07_23.csv"  # ścieżka do pliku z danymi
sep         = "\t"                            # separator (w Twoim pliku jest tab)
n_components_pca = 2                          # liczba komponentów PCA = liczba kubitów
test_size   = 0.20                            # ułamek danych do testu
random_state = 42                             # ziarno losowe
maxiter     = 20                             # iteracje optymalizatora
entanglement = "linear"                       # "linear" | "full" | lista par
reps_feature = 2                              # głębokość feature map
reps_ansatz  = 2  

In [3]:
def build_qnn(num_features: int):
    """
    Buduje EstimatorQNN z ZZFeatureMap i TwoLocal.
    """
    feature_map = ZZFeatureMap(feature_dimension=num_features, reps=reps_feature, entanglement=entanglement)
    ansatz = TwoLocal(
        num_qubits=n_components_pca,
        reps=reps_ansatz,
        rotation_blocks=["ry", "rz"],
        entanglement_blocks="cz",
        entanglement=entanglement,
    )
    estimator = Estimator()  # Aer backend

    qnn = EstimatorQNN(
        circuit=feature_map.compose(ansatz),
        input_params=feature_map.parameters,
        weight_params=ansatz.parameters,
        estimator=estimator,
    )

    optimizer = COBYLA(maxiter=maxiter)   # tutaj ustawiamy maxiter
    clf = NeuralNetworkClassifier(
        neural_network=qnn,
        optimizer=optimizer,
        one_hot=False,    # bo etykiety to 0/1
    )
    return clf

In [4]:
# ===== 1) Wczytanie i przygotowanie danych =====
print("Wczytywanie danych...")
df = pd.read_csv(data_path, sep=sep)
df = df.T

Wczytywanie danych...


In [5]:
metadata = pd.read_csv("SampleInfo_fixed_08_07_23.csv", delimiter=";")
metadata = metadata.set_index("id")
metadata["label"] = metadata["GroupAlternative"].apply(
    lambda x: 0 if x == "Asymptomatic controls" else 1
)
metadata = metadata[metadata["RealLocation"] != "Institute 5"]
df = df.merge(metadata, left_index=True, right_index=True)

In [6]:
X = df.drop(columns=metadata.columns)
y = df["label"]

In [ ]:
print("Shape X:", X.shape)
print("Shape y:", y.shape)
print("Class balance:\n", y.value_counts())
print(f"Liczba cech (genów): {X.shape[1]}, liczba próbek: {X.shape[0]}")
print(f"Klasy: 0 (zdrowe) = {(y==0).sum()}, 1 (nowotworowe) = {(y==1).sum()}")

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=test_size, random_state=random_state, stratify=y
)

In [ ]:
# Najpierw wybierz 100 najlepiej różnicujących cech
selector = SelectKBest(score_func=f_classif, k=100)
X_selected = selector.fit_transform(X, y)

scaler = StandardScaler()
X_train_std = scaler.fit_transform(X_train)
X_test_std  = scaler.transform(X_test)

# Dopiero potem licz PCA na tych cechach
pca = PCA(n_components=n_components_pca, random_state=random_state)
X_train_pca = pca.fit_transform(X_train_std)
X_test_pca  = pca.transform(X_test_std)
print(f"Po PCA: X_train = {X_train_pca.shape}, X_test = {X_test_pca.shape}")

In [ ]:
X_train_qnn = X_train_pca
X_test_qnn = X_test_pca
y_train_qnn = y_train.to_numpy()
y_test_qnn = y_test.to_numpy()

In [ ]:
t0 = time.time()
qnn = build_qnn(num_features=n_components_pca)
qnn.fit(X_train_pca, y_train)
y_pred = qnn.predict(X_test_pca)
acc_qnn = accuracy_score(y_test, y_pred)
t_qnn = time.time() - t0

In [ ]:
y_pred_train = qnn.predict(X_train_pca)

# 1. Spłaszczenie predykcji
y_pred_train = y_pred_train.ravel()

# 2. Zamiana -1 na 0 (bo y_train ma tylko 0 i 1)
y_pred_train = np.where(y_pred_train == -1, 0, y_pred_train)

# 3. Konwersja do int
y_train = y_train.astype(int)
y_pred_train = y_pred_train.astype(int)

# 4. Obliczanie metryk
acc_train = accuracy_score(y_train, y_pred_train)
bal_acc_train = balanced_accuracy_score(y_train, y_pred_train)
sens_train = recall_score(y_train, y_pred_train, pos_label=1)  # TPR
spec_train = recall_score(y_train, y_pred_train, pos_label=0)  # TNR

try:
    y_score_train = vqc.predict_proba(X_train_pca)[:, 1]
    auc_train = roc_auc_score(y_train, y_score_train)
except Exception:
    auc_train = roc_auc_score(y_train, y_pred_train)

In [ ]:
print(f"Accuracy: {acc_train:.3f}")
print(f"Balanced Accuracy: {bal_acc_train:.3f}")
print(f"Sensitivity (TPR): {sens_train:.3f}")
print(f"Specificity (TNR): {spec_train:.3f}")
print(f"ROC AUC:            {auc_train:.4f}")

In [ ]:
y_pred_test = qnn.predict(X_test_pca)

# 1. Spłaszczenie predykcji
y_pred_test = y_pred_test.ravel()

# 2. Zamiana -1 na 0 (bo y_train ma tylko 0 i 1)
y_pred_test = np.where(y_pred_test == -1, 0, y_pred_test)

# 3. Konwersja do int
y_test = y_test.astype(int)
y_pred_test = y_pred_test.astype(int)

# 4. Obliczanie metryk
acc_test = accuracy_score(y_test, y_pred_test)
bal_acc_test = balanced_accuracy_score(y_test, y_pred_test)
sens_test = recall_score(y_test, y_pred_test, pos_label=1)  # TPR
spec_test = recall_score(y_test, y_pred_test, pos_label=0)  # TNR

try:
    y_score_test = qnn.predict_proba(X_test_pca)[:, 1]
    auc_test = roc_auc_score(y_test, y_score_test)
except Exception:
    auc_test = roc_auc_score(y_test, y_pred_test)

In [ ]:
print(f"Accuracy: {acc_test:.3f}")
print(f"Balanced Accuracy: {bal_acc_test:.3f}")
print(f"Sensitivity (TPR): {sens_test:.3f}")
print(f"Specificity (TNR): {spec_test:.3f}")
print(f"ROC AUC:            {auc_test:.4f}")

In [ ]:
print("\n=== Wyniki na TRAIN ===")
print(f"Accuracy:           {acc_train:.4f}")
print(f"Balanced accuracy:  {bal_acc_train:.4f}")
print(f"Sensitivity (TPR):  {sens_train:.4f}")
print(f"Specificity (TNR):  {spec_train:.4f}")
print(f"ROC AUC:            {auc_train:.4f}")
print("\n=== Wyniki na TEST ===")
print(f"Accuracy:           {acc_test:.4f}")
print(f"Balanced accuracy:  {bal_acc_test:.4f}")
print(f"Sensitivity (TPR):  {sens_test:.4f}")
print(f"Specificity (TNR):  {spec_test:.4f}")
print(f"ROC AUC:            {auc_test:.4f}")

In [ ]:
print(f"Czas wykonania (s): {t_qnn:.2f}")

In [ ]:
# ===== ROC curve dla TRAIN =====
try:
    y_score_train = vqc.predict_proba(X_train_pca)[:, 1]
except Exception:
    # jeżeli nie ma predict_proba, bierzemy etykiety (to da schodkową krzywą)
    y_score_train = y_pred_train

fpr_train, tpr_train, _ = roc_curve(y_train, y_score_train)

# ===== ROC curve dla TEST =====
try:
    y_score_test = vqc.predict_proba(X_test_pca)[:, 1]
except Exception:
    y_score_test = y_pred_test

fpr_test, tpr_test, _ = roc_curve(y_test, y_score_test)

In [ ]:
# ===== Rysowanie =====
plt.figure(figsize=(6, 5))
plt.plot(fpr_train, tpr_train, label=f"Train (AUC={auc_train:.3f})")
plt.plot(fpr_test, tpr_test, label=f"Test (AUC={auc_test:.3f})")

plt.plot([0, 1], [0, 1], "k--", label="Random")
plt.xlabel("False Positive Rate (1 - Specificity)")
plt.ylabel("True Positive Rate (Sensitivity)")
plt.title("ROC Curve")
plt.legend(loc="lower right")
plt.grid(True, linestyle="--", alpha=0.7)
plt.show()